02:
CSV篩選事故車 畫trajectory

#流程
df_B->df_precrash:事故前、事故車 B 的原始 tracking 資料
->找 first_truncated_frame
->檢查 truncation 前的 bbox 異常
->df_corrected:修正異常 bbox 後的資料
->df_ttc:再切掉 truncation 後，只留下可計算 TTC 的資料
->scale
->slope_15
->ttc
positive_ttc#slope_15 > 0、TTC 有意義的區間

In [ ]:
#從本機上傳tracking完的csv
import pandas as pd

df = pd.read_csv("/content/tracking_result_1.csv")
df.head()

In [ ]:
#看影片 找事故車的id(手動)
#留下track_id==事故車
#影片1為id=2 把該id的資訊copy一份
B_ID = 2
df_B = df[df["track_id"] == B_ID].copy()

print(df_B.shape)
print(df_B.head())
print(df_B.tail())

In [ ]:
#看B第一次/最後一次出現的frame 對比總幀數 確認B是否在某些frame消失
#若消失 問題可能來自detection miss、tracker問題、遮擋...等等(先不要修)
print("First frame:", df_B["frame"].min())
print("Last frame:", df_B["frame"].max())
print("Number of detections:", len(df_B))

##確認B的bbox沒有miss

In [ ]:
#原始資料已有collition time,FPS皆=30
#找出撞擊當下約在哪一幀

FPS = 30
collision_time = 19.23 #影片1
collision_frame = round(collision_time * FPS)

print(collision_frame)

In [ ]:
#copy事故前的frame(切掉事故發生後的資訊)(不需要)
df_precrash = df_B[df_B["frame"] <= collision_frame].copy()

df_precrash["time"] = df_precrash["frame"] / FPS

# time_to_impact:距離碰撞還有幾秒
# 0 = collision
df_precrash["time_to_impact"] = (
    collision_time - df_precrash["time"]
)

df_precrash[[
    "frame", "time", "time_to_impact",
    "center_x", "center_y",
    "bbox_width", "bbox_height"
]].tail()

畫x,y中心點 以及bbox對時間的圖

In [ ]:
#center_x
fig, ax = plt.subplots()

ax.plot(
    df_precrash["time_to_impact"],
    df_precrash["center_x"]
)

ax.set_xlabel("Time to Impact (s)")
ax.set_ylabel("Center X (pixels)")
ax.set_title("Baseline Center X")

ax.invert_xaxis()
plt.show()

In [ ]:
#center_y
fig, ax = plt.subplots()

ax.plot(
    df_precrash["time_to_impact"],
    df_precrash["center_y"]
)

ax.set_xlabel("Time to Impact (s)")
ax.set_ylabel("Center Y (pixels)")
ax.set_title("Baseline Center Y")

ax.invert_xaxis()
plt.show()

In [ ]:
#bounding box width
fig, ax = plt.subplots()

ax.plot(
    df_precrash["time_to_impact"],
    df_precrash["bbox_width"]
)

ax.set_xlabel("Time to Impact (s)")
ax.set_ylabel("Bounding Box Width (pixels)")
ax.set_title("Baseline Bounding Box Width")

ax.invert_xaxis()
plt.show()

In [ ]:
#bounding box height
fig, ax = plt.subplots()

ax.plot(
    df_precrash["time_to_impact"],
    df_precrash["bbox_height"]
)

ax.set_xlabel("Time to Impact (s)")
ax.set_ylabel("Bounding Box Height (pixels)")
ax.set_title("Baseline Bounding Box Height")

ax.invert_xaxis()
plt.show()

#確認車身超出鏡頭的幀數(去除該段影片)


In [ ]:
#確認從哪一幀開始 由於相機太靠近前車 導致前車完整車身超出鏡頭、bbox判定失準
#尋找大概的區間
for f in range(510, 578, 10):
    show_bbox(f)

In [ ]:
#尋找確切幀數->得frame 541(開始出現明顯 truncation 的位置)
for f in range(535, 546):
    show_bbox(f)

In [ ]:
#df_ttc:排除「前車開始超出畫面」後的資料 用來計算 TTC
first_truncated_frame = 541
last_valid_frame = 540



#修復框歪的bbox

In [ ]:
#df_reference = 準備找出框歪的bbox的df
df_reference = df_precrash.copy()

In [ ]:
#先用 bbox_height 做，因為你目前最在意的是 height 突然框錯會影響 TTC

# 1. 建立局部中位數趨勢
df_reference["height_median"] = (
    df_reference["bbox_height"]
    .rolling(window=15, center=True) #這一幀前後約半秒(window=FPS/2)的 bbox 通常多高？
    .median() #取這 15 個數字的中位數 median
)

# 2. 這一幀的 bbox height 距離附近的典型 height 多遠？(局部趨勢差多少)
df_reference["height_deviation"] = abs(
    df_reference["bbox_height"]
    - df_reference["height_median"]
)


# 3. 計算每一幀相對於附近典型 height，偏差了多少比例
# 看整體分布 決定threshold
df_reference["height_relative_deviation"] = (
    df_reference["height_deviation"]
    / df_reference["height_median"]
)

# 看所有 frames 的相對偏差分布
df_reference["height_relative_deviation"].describe()



In [ ]:
#畫所有 frames 的分布
plt.hist(
    df_reference["height_relative_deviation"].dropna(),
    bins=50
)

plt.xlabel("Relative Height Deviation")
plt.ylabel("Number of Frames")
plt.title("Distribution of BBox Height Deviation")
plt.show()

In [ ]:
#把尾巴上的 frames 全部抓出來人工檢查。
#先暫時用 2% 當「人工檢查門檻」，注意它不是「錯誤門檻」。
check_frames = df_reference[
    df_reference["height_relative_deviation"] >= 0.02
][
    ["frame",
     "bbox_height",
     "height_median",
     "height_relative_deviation"]
]

check_frames

In [ ]:
len(check_frames) #看長度決定是否逐幀檢查

In [ ]:
#def一函式 把特定 frame + YOLO bbox 顯示出來
import cv2
def show_bbox(frame_number):

    cap = cv2.VideoCapture(VIDEO_PATH)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        print("Could not read frame")
        return

    row = df_reference[
        df_reference["frame"] == frame_number
    ].iloc[0]

    x1 = int(row["x1"])
    y1 = int(row["y1"])
    x2 = int(row["x2"])
    y2 = int(row["y2"])

    cv2.rectangle(
        frame,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        3
    )

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 7))
    plt.imshow(frame_rgb)
    plt.title(f"Frame {frame_number}")
    plt.axis("off")
    plt.show()

In [ ]:
check_frames["frame"].tolist()

In [ ]:
#畫出所有可疑frame
VIDEO_PATH = "/content/1.mp4"
for f in check_frames["frame"]:
    show_bbox(int(f))

In [ ]:
#看可疑的frame前後的frame是否也跑掉
#如果沒跑掉 可以插值 補那些跑掉的frame
for f in [555,556,557,558]:
    show_bbox(f)

In [ ]:
bad_frames = list(range(548, 558))#手動抓的有問題的frame:548-557
"""
中間只有 10 frames(約0.3秒)
由於時間極短,車子的bbox合理地可以先用平滑變化近似
"""

In [ ]:
#df_reference = 準備找出框歪的bbox的df
#df_corrected: 人工修正框歪的bbox 用547和558插值
df_corrected = df_reference.copy()

In [ ]:
cols = ["x1", "y1", "x2", "y2"]

# 把 548~557 的錯誤 bbox 暫時設成 NaN
df_corrected.loc[
    df_corrected["frame"].between(548, 557),cols
] = float("nan")

# 利用前後正常 frame 547、558 做線性插值
df_corrected[cols] = df_corrected[cols].interpolate(method="linear")

In [ ]:
#1. 重新計算修正後的 bbox height
df_corrected["bbox_height"] = (
    df_corrected["y2"] - df_corrected["y1"]
)

In [ ]:
#畫原始 vs 修正後 bbox height
import matplotlib.pyplot as plt

plot_range = (df_reference["frame"] >= 530) & (df_reference["frame"] <= 570)
plt.figure(figsize=(12, 6))

# 原始 YOLO bbox height
plt.plot(
    df_reference.loc[plot_range, "frame"],
    df_reference.loc[plot_range, "bbox_height"],
    label="Original BBox Height"
)

# 修正後 bbox height
plt.plot(
    df_corrected.loc[plot_range, "frame"],
    df_corrected.loc[plot_range, "bbox_height"],
    label="Corrected BBox Height"
)

plt.xlabel("Frame")
plt.ylabel("BBox Height (pixels)")
plt.title("Original vs Corrected BBox Height")
plt.legend()
plt.grid()

plt.show()

In [ ]:
#以數值檢查bbox是否調整成功
comparison = df_reference.loc[
    df_reference["frame"].between(547, 558),
    ["frame", "bbox_height"]
].copy()

comparison["corrected_height"] = df_corrected.loc[
    df_corrected["frame"].between(547, 558),
    "bbox_height"
].values

comparison

In [ ]:
#def 一函式 畫原始 vs 修正後 bbox
def show_original_vs_corrected(frame_number):
    # 讀取指定 frame
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        print("Could not read frame")
        return

    # 取得原始 YOLO bbox
    original = df_reference[
        df_reference["frame"] == frame_number
    ].iloc[0]

    # 取得 interpolation 後 bbox
    corrected = df_corrected[
        df_corrected["frame"] == frame_number
    ].iloc[0]

    # 原始 bbox 座標
    ox1 = int(original["x1"])
    oy1 = int(original["y1"])
    ox2 = int(original["x2"])
    oy2 = int(original["y2"])

    # 修正後 bbox 座標
    cx1 = int(corrected["x1"])
    cy1 = int(corrected["y1"])
    cx2 = int(corrected["x2"])
    cy2 = int(corrected["y2"])

    # 畫原始 bbox
    cv2.rectangle(
        frame,
        (ox1, oy1),
        (ox2, oy2),
        (0, 0, 255),
        3
    )

    # 畫修正後 bbox
    cv2.rectangle(
        frame,
        (cx1, cy1),
        (cx2, cy2),
        (0, 255, 0),
        3
    )

    # 標籤
    cv2.putText(
        frame, "Original",
        (ox1, max(oy1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7, (0, 0, 255), 2
    )

    cv2.putText(
        frame, "Corrected",
        (cx1, max(cy1 - 30, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7, (0, 255, 0), 2
    )

    # OpenCV BGR → matplotlib RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 7))
    plt.imshow(frame_rgb)
    plt.title(f"Frame {frame_number}")
    plt.axis("off")
    plt.show()

In [ ]:
#畫原始 vs 修正後 bbox
for f in range(548, 558):
    show_original_vs_corrected(f)

In [ ]:
#1. 先重新計算修正後的 x、y、w、h 因為我們修的是 x1, y1, x2, y2，所以這四個衍生值都要重新算。
# bbox center
df_corrected["center_x"] = (
    df_corrected["x1"] + df_corrected["x2"]
) / 2

df_corrected["center_y"] = (
    df_corrected["y1"] + df_corrected["y2"]
) / 2

# bbox width / height
df_corrected["bbox_width"] = (
    df_corrected["x2"] - df_corrected["x1"]
)

df_corrected["bbox_height"] = (
    df_corrected["y2"] - df_corrected["y1"]
)

In [ ]:
#畫修正完的bbox trajectory
import matplotlib.pyplot as plt

variables = [
    ("center_x", "Center X (pixels)"),
    ("center_y", "Center Y (pixels)"),
    ("bbox_width", "BBox Width (pixels)"),
    ("bbox_height", "BBox Height (pixels)")
]

for column, ylabel in variables:

    plt.figure(figsize=(12, 5))

    plt.plot(
        df_corrected["time"],
        df_corrected[column]
    )

    plt.xlabel("Time (seconds)")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} vs Time")

    plt.grid()
    plt.show()

In [ ]:
df_ttc = df_corrected[
    df_corrected["frame"] <= last_valid_frame
].copy()
df_ttc[[
    "frame",
    "time",
    "bbox_width",
    "bbox_height"
]].tail(10)

#開始計算s.s dot 畫TTC

同時利用 width 和 height，作為一個比較整體的 image scale。
避免w或h其中一個突然跑掉


In [ ]:
import numpy as np

df_ttc["scale"] = np.sqrt(
    df_ttc["bbox_width"] * df_ttc["bbox_height"]
)

In [ ]:
#畫圖看image sca(sqrt(wh))le vs Time
plt.figure(figsize=(12,5))

plt.plot(
    df_ttc["time"],
    df_ttc["scale"]
)

plt.xlabel("Time (seconds)")
plt.ylabel("Image Scale (pixels)")
plt.title("Image Scale vs Time")
plt.grid()

plt.show()

In [ ]:
#處理scale的小抖動
"""
TTC=s/dot s 分母亂將使TTC大亂
->先把 scale 平滑化
"""
df_ttc["scale_smooth"] = (
    df_ttc["scale"]
    .rolling(window=5, center=True) #取前後5幀(沒有理由?)
    .median()
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    df_ttc["time"],
    df_ttc["scale"],
    label="Original Scale"
)

plt.plot(
    df_ttc["time"],
    df_ttc["scale_smooth"],
    label="Smoothed Scale"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Image Scale (pixels)")
plt.title("Original vs Smoothed Image Scale")
plt.legend()
plt.grid()

plt.show()

In [ ]:
"""
1. 先算每秒 scale 改變多少
因為 FPS = 30： delta t = 1/30 sec
所以 dot{s}=s_t-s_{t-1}/{1/30}
"""
#前車在影像中的 scale 每秒正在增加多少 pixel。
df_ttc["scale_rate"] = (
    df_ttc["scale_smooth"].diff() * FPS
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    df_ttc["time"],
    df_ttc["scale_rate"]
)

plt.axhline(0)

plt.xlabel("Time (seconds)")
plt.ylabel("Scale Rate (pixels/second)")
plt.title("Image Scale Rate vs Time")
plt.grid()

plt.show()

In [ ]:
#scale rate劇烈抖動->推測由於.diff()每次只看兩幀 容易受到yolo noise干擾 因此dot{s} 一直在變。
"""
但其實從整體看：
154 ┤              ●
153 ┤          ●
152 ┤      ●
151 ┤   ●
150 ┤ ●
    └────────────────
     100 101 102 103 104

很明顯就是一條「大致往上」的線。
"""
#真正在乎:這一小段時間，scale 整體以多快的速度增加？
#所以可以拿這n個點，找一條最符合它們的直線：

#比較 5 / 15 / 30 frames 決定最適合的n

In [ ]:
#用np.polyfit() 在每個小 window 裡找最符合資料的直線。

def rolling_slope(df, window):

    slopes = []

    half = window // 2

    for i in range(len(df)):

        # window 超出資料範圍
        if i < half or i >= len(df) - half:
            slopes.append(np.nan)
            continue

        # 取目前 frame 前後的資料
        local = df.iloc[i-half : i+half+1]

        x = local["time"].values
        y = local["scale"].values

        # 找最符合這些點的直線 y = ax + b
        a, b = np.polyfit(x, y, 1)

        # a 就是 slope = ds/dt
        slopes.append(a)

    return slopes

In [ ]:
#5 / 15 / 30 frames
#window 越大 → 越穩，但反應越慢。
#window 越小 → 越即時，但越容易受 noise 影響。
df_ttc["slope_5"] = rolling_slope(df_ttc, 5)
df_ttc["slope_15"] = rolling_slope(df_ttc, 15)
df_ttc["slope_30"] = rolling_slope(df_ttc, 30)

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(df_ttc["time"], df_ttc["slope_5"], label="5 frames")
plt.plot(df_ttc["time"], df_ttc["slope_15"], label="15 frames")
plt.plot(df_ttc["time"], df_ttc["slope_30"], label="30 frames")

plt.axhline(0)

plt.xlabel("Time (seconds)")
plt.ylabel("Scale Rate (pixels/second)")
plt.title("Rolling Slope Comparison")
plt.legend()
plt.grid()

plt.show()

In [ ]:
"""
15-frame window was selected as a pilot setting to reduce frame-level bounding-box noise
while retaining the rapid scale changes during the pre-collision closing phase.

我們已經有：[s=\sqrt{wh}]
[dot{s}=\text{slope\_15}]

但只有：[\dot{s}>0]
也就是前車正在影像中擴張時，才計算正 TTC
"""

In [ ]:
df_ttc["ttc"] = np.where(
    df_ttc["slope_15"] > 0,
    df_ttc["scale"] / df_ttc["slope_15"],
    np.nan
)

In [ ]:
#先只畫 14 秒以後，這樣比較容易看真正的 closing phase：
closing = df_ttc[df_ttc["time"] >= 14]

plt.figure(figsize=(12,5))

plt.plot(
    closing["time"],
    closing["ttc"]
)

plt.xlabel("Time (seconds)")
plt.ylabel("TTC (seconds)")
plt.title("Baseline TTC vs Time")
plt.ylim(0, 10)
plt.grid()

plt.show()

In [ ]:
#實際距離碰撞還剩多久
df_ttc["actual_time_to_impact"] = (
    collision_time - df_ttc["time"]
)

In [ ]:
#接著畫在一起。先看 16.5 秒以後：
comparison = df_ttc[
    df_ttc["time"] >= 16.5
]

plt.figure(figsize=(12,5))

# 我們用 bbox expansion 估計的 TTC
plt.plot(
    comparison["time"],
    comparison["ttc"],
    label="Estimated TTC"
)

# 根據實際 collision time 算出的剩餘時間
plt.plot(
    comparison["time"],
    comparison["actual_time_to_impact"],
    label="Actual Time to Impact"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Time (seconds)")
plt.title("Estimated TTC vs Actual Time to Impact")

plt.ylim(0, 10)
plt.legend()
plt.grid()

plt.show()